# Prudential Life Insurance Assessment


<img src='https://www.sigortahakimi.com/wp-content/uploads/hayat-sigortasi-sigortahakimi.jpeg'>


Bu çalışmada `Prudential Life Insurance Assessment` yarışması için müşteri sağlık ve sigorta özellikleri kullanılarak risk değerlendirme sınıfını tahmin eden bir başlangıç modeli oluşturulmuştur.


## Akış

1. Kütüphaneleri yükleme
2. Drive bağlama ve zip içeriğini tanımlama
3. Veri dosyalarını yükleme
4. Ön işleme
5. Özellik çıkarımı
6. Model kurma
7. Accuracy değerlendirmesi
8. Test tahmini ve submission
9. Sonuç


## 1. Kütüphaneleri Yükleme


In [1]:
!pip install catboost -q


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 8.3 MB/s eta 0:00:00


In [2]:
from google.colab import drive
import io
import os
import zipfile

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from catboost import CatBoostClassifier

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)


## 2. Drive Bağlama ve Zip İçeriğini Tanımlama


In [3]:
# Bu bölümde yarışma zip dosyasını Drive üzerinden bağlıyor ve gerekli dosyaları buluyoruz.


In [4]:
drive.mount('/content/drive')

zip_path = '/content/drive/MyDrive/Colab Data Dosyaları/prudential-life-insurance-assessment.zip'
if not os.path.exists(zip_path):
    raise FileNotFoundError('Zip dosyası bulunamadı. Colab Data Dosyaları içindeki gerçek dosya adını kontrol et.')

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_members = zip_ref.namelist()

def find_zip_member(members, candidates):
    for candidate in candidates:
        for member in members:
            if member.endswith(candidate):
                return member
    raise FileNotFoundError(f'{candidates[0]} zip içinde bulunamadi.')

train_member = find_zip_member(zip_members, ['train.csv', 'train.csv.zip'])
test_member = find_zip_member(zip_members, ['test.csv', 'test.csv.zip'])
sample_submission_member = find_zip_member(zip_members, ['sample_submission.csv', 'sample_submission.csv.zip'])


Mounted at /content/drive


## 3. Veri Dosyalarını Yükleme


In [5]:
# Bu bölümde yarışma dosyalarını doğrudan zip içinden yüklüyoruz.


In [6]:
def read_csv_from_zip(zip_path, member_name):
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        with zip_ref.open(member_name) as f:
            if member_name.endswith('.zip'):
                inner_bytes = f.read()
                with zipfile.ZipFile(io.BytesIO(inner_bytes), 'r') as inner_zip:
                    inner_name = inner_zip.namelist()[0]
                    with inner_zip.open(inner_name) as inner_f:
                        return pd.read_csv(inner_f)
            return pd.read_csv(f)

train = read_csv_from_zip(zip_path, train_member)
test = read_csv_from_zip(zip_path, test_member)
sample_submission = read_csv_from_zip(zip_path, sample_submission_member)

print('Train shape:', train.shape)
print('Test shape:', test.shape)
print('Sample submission shape:', sample_submission.shape)
train.head()


Train shape: (59381, 128)
Test shape: (19765, 127)
Sample submission shape: (19765, 2)


,Id,Product_Info_1,Product_Info_2,Product_Info_3,Product_Info_4,Product_Info_5,Product_Info_6,Product_Info_7,Ins_Age,Ht,Wt,BMI,Employment_Info_1,Employment_Info_2,Employment_Info_3,Employment_Info_4,Employment_Info_5,Employment_Info_6,InsuredInfo_1,InsuredInfo_2,InsuredInfo_3,InsuredInfo_4,InsuredInfo_5,InsuredInfo_6,InsuredInfo_7,Insurance_History_1,Insurance_History_2,Insurance_History_3,Insurance_History_4,Insurance_History_5,Insurance_History_7,Insurance_History_8,Insurance_History_9,Family_Hist_1,Family_Hist_2,Family_Hist_3,Family_Hist_4,Family_Hist_5,Medical_History_1,Medical_History_2,Medical_History_3,Medical_History_4,Medical_History_5,Medical_History_6,Medical_History_7,Medical_History_8,Medical_History_9,Medical_History_10,Medical_History_11,Medical_History_12,Medical_History_13,Medical_History_14,Medical_History_15,Medical_History_16,Medical_History_17,Medical_History_18,Medical_History_19,Medical_History_20,Medical_History_21,Medical_History_22,Medical_History_23,Medical_History_24,Medical_History_25,Medical_History_26,Medical_History_27,Medical_History_28,Medical_History_29,Medical_History_30,Medical_History_31,Medical_History_32,Medical_History_33,Medical_History_34,Medical_History_35,Medical_History_36,Medical_History_37,Medical_History_38,Medical_History_39,Medical_History_40,Medical_History_41,Medical_Keyword_1,Medical_Keyword_2,Medical_Keyword_3,Medical_Keyword_4,Medical_Keyword_5,Medical_Keyword_6,Medical_Keyword_7,Medical_Keyword_8,Medical_Keyword_9,Medical_Keyword_10,Medical_Keyword_11,Medical_Keyword_12,Medical_Keyword_13,Medical_Keyword_14,Medical_Keyword_15,Medical_Keyword_16,Medical_Keyword_17,Medical_Keyword_18,Medical_Keyword_19,Medical_Keyword_20,Medical_Keyword_21,Medical_Keyword_22,Medical_Keyword_23,Medical_Keyword_24,Medical_Keyword_25,Medical_Keyword_26,Medical_Keyword_27,Medical_Keyword_28,Medical_Keyword_29,Medical_Keyword_30,Medical_Keyword_31,Medical_Keyword_32,Medical_Keyword_33,Medical_Keyword_34,Medical_Keyword_35,Medical_Keyword_36,Medical_Keyword_37,Medical_Keyword_38,Medical_Keyword_39,Medical_Keyword_40,Medical_Keyword_41,Medical_Keyword_42,Medical_Keyword_43,Medical_Keyword_44,Medical_Keyword_45,Medical_Keyword_46,Medical_Keyword_47,Medical_Keyword_48,Response
0,2,1,D3,10,0.076923,2,1,1,0.641791,0.581818,0.148536,0.323008,0.028,12,1,0.0,3,NaN,1,2,6,3,1,2,1,1,1,3,1,0.000667,1,1,2,2,NaN,0.598039,NaN,0.526786,4.0,112,2,1,1,3,2,2,1,NaN,3,2,3,3,240.0,3,3,1,1,2,1,2,3,NaN,1,3,3,1,3,2,3,NaN,1,3,1,2,2,1,3,3,3,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,8
1,5,1,A1,26,0.076923,2,3,1,0.059701,0.600000,0.131799,0.272288,0.000,1,3,0.0,2,0.0018,1,2,6,3,1,2,1,2,1,3,1,0.000133,1,3,2,2,0.188406,NaN,0.084507,NaN,5.0,412,2,1,1,3,2,2,1,NaN,3,2,3,3,0.0,1,3,1,1,2,1,2,3,NaN,1,3,3,1,3,2,3,NaN,3,1,1,2,2,1,3,3,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,4
2,6,1,E1,26,0.076923,2,3,1,0.029851,0.745455,0.288703,0.428780,0.030,9,1,0.0,2,0.0300,1,2,8,3,1,1,1,2,1,1,3,NaN,3,2,3,3,0.304348,NaN,0.225352,NaN,10.0,3,2,2,1,3,2,2,2,NaN,3,2,3,3,NaN,1,3,1,1,2,1,2,3,NaN,2,2,3,1,3,2,3,NaN,3,3,1,3,2,1,3,3,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,8
3,7,1,D4,10,0.487179,2,3,1,0.164179,0.672727,0.205021,0.352438,0.042,9,1,0.0,3,0.2000,2,2,8,3,1,2,1,2,1,1,3,NaN,3,2,3,3,0.420290,NaN,0.352113,NaN,0.0,350,2,2,1,3,2,2,2,NaN,3,2,3,3,NaN,1,3,1,1,2,2,2,3,NaN,1,3,3,1,3,2,3,NaN,3,3,1,2,2,1,3,3,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,8
4,8,1,D2,26,0.230769,2,3,1,0.417910,0.654545,0.234310,0.424046,0.027,9,1,0.0,2,0.0500,1,2,6,3,1,2,1,2,1,1,3,NaN,3,2,3,2,0.463768,NaN,0.408451,NaN,NaN,162,2,2,1,3,2,2,2,NaN,3,2,3,3,NaN,1,3,1,1,2,1,2,3,NaN,2,2,3,1,3,2,3,NaN,3,3,1,3,2,1,3,3,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,8


## 4. Ön İşleme


In [7]:
# Bu bölümde hedef değişkeni ayırıyor ve temel temizlik adımlarını uyguluyoruz.


In [8]:
target_col = 'Response'

train = train.replace([np.inf, -np.inf], np.nan)
test = test.replace([np.inf, -np.inf], np.nan)

feature_cols = [col for col in train.columns if col not in ['Id', target_col] and col in test.columns]


## 5. Özellik Çıkarımı


In [9]:
# Bu bölümde sayısal özellikleri hazırlıyor ve eğitim/validation ayrımı yapıyoruz.


In [10]:
train_x = train[feature_cols].copy()
train_y = train[target_col].copy()
test_x = test[feature_cols].copy()

for col in feature_cols:
    train_x[col] = pd.to_numeric(train_x[col], errors='coerce')
    test_x[col] = pd.to_numeric(test_x[col], errors='coerce')

x_train, x_valid, y_train, y_valid = train_test_split(
    train_x, train_y, test_size=0.2, random_state=42, stratify=train_y
)

print('x_train shape:', x_train.shape)
print('x_valid shape:', x_valid.shape)


x_train shape: (47504, 126)
x_valid shape: (11877, 126)


## 6. Model Kurma


In [11]:
# Bu bölümde risk değerlendirme sınıfı tahmini için CatBoost modelini kuruyoruz.


In [12]:
model = CatBoostClassifier(
    iterations=300,
    learning_rate=0.08,
    depth=8,
    loss_function='MultiClass',
    eval_metric='Accuracy',
    verbose=0,
    random_seed=42
)

model.fit(
    x_train,
    y_train,
    eval_set=(x_valid, y_valid),
    use_best_model=True
)


CatBoostClassifier(depth=8, eval_metric='Accuracy', iterations=300, learning_rate=0.08, loss_function='MultiClass', random_seed=42, verbose=0)

## 7. Accuracy Değerlendirmesi


In [13]:
# Bu bölümde validation verisi üzerindeki accuracy değerini hesaplıyoruz.


In [14]:
valid_preds = model.predict(x_valid).astype(int).flatten()
acc = accuracy_score(y_valid, valid_preds)
print('Validation Accuracy:', round(acc, 5))


Validation Accuracy: 0.58289


## 8. Test Tahmini ve Submission


In [15]:
# Bu bölümde test tahminlerini submission formatına dönüştürüyoruz.


In [16]:
test_preds = model.predict(test_x).astype(int).flatten()
submission = sample_submission.copy()
if 'Response' in submission.columns:
    submission['Response'] = test_preds[:len(submission)]
else:
    submission.iloc[:, 1] = test_preds[:len(submission)]

print('Submission shape:', submission.shape)
submission.head()


Submission shape: (19765, 2)


,Id,Response
0,1,1
1,3,8
2,4,6
3,9,8
4,12,8


In [17]:
output_path = '/content/prudential_submission.csv'
submission.to_csv(output_path, index=False)
print('Kaydedilen dosya:', output_path)


Kaydedilen dosya: /content/prudential_submission.csv


## 9. Sonuç


Bu çalışmada Prudential Life Insurance Assessment yarışması için müşteri sağlık ve sigorta özellikleri kullanılarak risk değerlendirme sınıfını tahmin eden bir başlangıç modeli oluşturuldu. Elde edilen sonuçlara göre model validation verisi üzerinde 0.58289 accuracy değeri elde etti ve test verisi için risk sınıfı tahminleri üretildi.
